# EPINUC colocalization — counts per FOV per sample

This notebook uses the importable `epinuc_colocalization.py` module and runs the exact same
pipeline as `demo_epinuc_colocalization.ipynb`, but reports the counts **per field of view (FOV)
per sample** instead of one summed row per sample.

For each type — nucleosomes, R PTMs, B PTMs, and the R–nucleosome / B–nucleosome (co),
R–B (bi), and triple (tri) colocalizations — we report the **final cumulative de-duplicated
count of each FOV**. Timepoints are still collapsed (each FOV's cumulative count over time is
the de-duplicated total across all its timepoints), but we do **not** sum over FOVs.

The stock `ep.sample_summary_frame` computes `cum.groupby('scene').max()` and then sums those
per-FOV finals across FOVs. This notebook keeps that per-FOV table and skips the final sum.

Requirements: `epinuc_colocalization.py` next to this notebook, and the ND2 files in a folder
(default `./T50_20260225`).

In [ ]:
import os
import pandas as pd
import epinuc_colocalization as ep

DATA_DIR = "./T50_20260225"          # TODO: folder with your ND2 files
OUTPUT_DIR = "./per_fov_output"      # where the per-FOV CSV is written

# --- Sample names -------------------------------------------------------------------------
# The ND2 filenames only carry integer sample ids (TS<timepoint>_<sample>.nd2 -> our `sample`,
# which equals CellProfiler's Metadata_Lane). The lane->name mapping below comes from the
# reference summary  T50_20260225/output/EPINUC_sum_20260225_1.csv  (columns Sample, Metadata_Lane).
# Any id left out falls back to its number. These names label every output table (a
# `sample_name` column) and every plot axis.
SAMPLE_NAMES = {
    1: "L123",
    2: "LR143",
    3: "H_29mm",
    4: "L124",
    5: "LR144",
    6: "H_34mm",
}


def name_of(sid):
    """Display name for a sample id (falls back to the number as a string)."""
    return SAMPLE_NAMES.get(int(sid), str(sid))


# Which samples are present? (parses TS<timepoint>_<sample>.nd2)
by_sample = ep.group_files_by_sample(ep.list_nd2_files(DATA_DIR))
print("Available samples ->",
      {name_of(s): len(v) for s, v in sorted(by_sample.items()) if s is not None})

## Per-FOV summary helpers

`per_fov_summary_frame` is the per-FOV analogue of `ep.sample_summary_frame`: it takes the
module's cumulative table (`acc.cumulative`) and, for each FOV (`scene`), reports the final
cumulative count of each type. Because a FOV's cumulative count is monotonic in time, that
final value is the per-FOV max over timepoints — i.e. timepoints are collapsed. The only
difference from the stock summary is that we keep one row per FOV instead of summing them.

In [ ]:
# The de-duplicated cumulative count columns, reused from the module so this stays in sync.
CUM_TYPE_COLS = ep._CUM_TYPE_COLS

# Friendlier names for the colocalization columns (co / bi / tri).
RENAME = {
    "cumulative_n_nucleosomes": "n_nucleosomes",
    "cumulative_n_R_PTMs": "n_R_PTMs",
    "cumulative_n_B_PTMs": "n_B_PTMs",
    "cumulative_n_R_colocalized_with_nucleosome": "n_R_nucleosome_coloc",
    "cumulative_n_B_colocalized_with_nucleosome": "n_B_nucleosome_coloc",
    "cumulative_n_R_B_colocalized": "n_R_B_bicoloc",
    "cumulative_n_triple_colocalized": "n_triple_coloc",
}


def per_fov_summary_frame(acc, sample_id):
    """One row per FOV for a sample: final cumulative (de-duplicated) count of each type.

    Timepoints are collapsed (per-FOV cumulative max over time); FOVs are NOT summed.
    """
    cum = pd.DataFrame(acc.cumulative, columns=ep.CUM_COLUMNS)
    if len(cum) == 0:
        return pd.DataFrame(columns=["sample", "FOV", "n_timepoints", *CUM_TYPE_COLS])
    per_fov = cum.groupby("scene")[CUM_TYPE_COLS].max()          # final count per FOV
    n_tp = cum.groupby("scene")["time"].nunique()                 # timepoints seen per FOV
    out = per_fov.reset_index().rename(columns={"scene": "FOV"})
    out.insert(0, "sample", sample_id)
    out.insert(2, "n_timepoints", out["FOV"].map(n_tp).astype(int))
    return out.astype({c: int for c in CUM_TYPE_COLS}).sort_values("FOV").reset_index(drop=True)


def run_samples_per_fov(samples, data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
                        n_jobs=None, scenes=None, write_details=False,
                        summary_path=None, rename=True):
    """Run one or more samples and return counts per FOV per sample (one row per FOV).

    Mirrors ``ep.run_samples`` but keeps the per-FOV breakdown instead of summing over FOVs.
    """
    if isinstance(samples, int):
        samples = [samples]
    os.makedirs(output_dir, exist_ok=True)
    all_files = ep.list_nd2_files(data_dir)
    grouped = ep.group_files_by_sample(all_files)
    frames = []
    for sid in samples:
        if sid not in grouped:
            print(f"!! sample {sid} not found in {data_dir} (have {sorted(k for k in grouped if k is not None)})")
            continue
        print(f"\n=== Sample {sid}: {len(grouped[sid])} timepoint(s) ===")
        acc = ep.process_sample_parallel(grouped[sid], sample_id=sid, n_jobs=n_jobs,
                                         scenes=scenes, export_dir=None)
        if write_details:
            ep.export_results(acc, output_dir=os.path.join(output_dir, f"sample_{sid}"))
        frames.append(per_fov_summary_frame(acc, sid))
    per_fov = (pd.concat(frames, ignore_index=True) if frames
               else pd.DataFrame(columns=["sample", "FOV", "n_timepoints", *CUM_TYPE_COLS]))
    if rename:
        per_fov = per_fov.rename(columns=RENAME)
    summary_path = summary_path or os.path.join(output_dir, "per_fov_counts.csv")
    per_fov.to_csv(summary_path, index=False)
    print(f"\nWrote per-FOV counts ({len(per_fov)} FOV row(s)) -> {summary_path}")
    return per_fov

## 1. One sample (quick test on the first 3 FOVs)

`scenes=range(3)` keeps this fast. Use `scenes=None` to process every FOV.

In [ ]:
df_one = run_samples_per_fov(
    1,                       # sample id (int) — or a list, see below
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    scenes=range(3),         # TODO: set to None for all FOVs
    n_jobs=ep.N_JOBS,        # -2 = all but one core; 1 = serial
)
df_one

## 2. Several samples at once

Pass a list of ids. Each **FOV** of each sample becomes one row in the returned table and in
`OUTPUT_DIR/per_fov_counts.csv`.

In [ ]:
ep.N_JOBS

In [ ]:
df = run_samples_per_fov(
    [1, 2, 3,4,5,6],               # TODO: your sample ids, e.g. sorted(k for k in by_sample if k is not None)
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    scenes=None,         # TODO: None for the full run
)
df

## 3. Full production run & extras

- **All FOVs:** set `scenes=None`.
- **Also keep the detailed tables** (per-spot, per-bead, colocalization events, per-image and
  cumulative counts): pass `write_details=True` — they go to `OUTPUT_DIR/sample_<id>/`.
- **Sanity check:** summing the per-FOV rows over FOV reproduces `ep.sample_summary_frame`'s
  one-row-per-sample totals (the cell below).

Columns: `n_nucleosomes`, `n_R_PTMs`, `n_B_PTMs`, `n_R_nucleosome_coloc` &
`n_B_nucleosome_coloc` (the R/B "co"-localizations with a nucleosome), `n_R_B_bicoloc` (R–B
"bi"), and `n_triple_coloc` ("tri"). `n_timepoints` is how many timepoints that FOV was seen in.

In [ ]:
# Full run for every available sample, all FOVs (uncomment):
# samples = sorted(k for k in by_sample if k is not None)
# df_all = run_samples_per_fov(samples, data_dir=DATA_DIR, output_dir=OUTPUT_DIR,
#                              scenes=None, write_details=True)
# df_all

In [ ]:
# Optional cross-check: per-FOV rows summed over FOV == the stock per-sample summary.
value_cols = list(RENAME.values())
per_sample_from_fov = df.groupby("sample", as_index=False)[value_cols].sum()
per_sample_from_fov

## 4. Per-nucleosome occupancy probabilities (per FOV)

Occupancy is a **per-nucleosome** property, so the denominator is the number of nucleosomes in
the FOV. **A = R (red) PTM, B = B (blue) PTM.** From the per-FOV cumulative counts:

- `N`   = `n_nucleosomes` — all nucleosomes (de-duplicated over timepoints)
- `nA`  = `n_R_nucleosome_coloc` — nucleosomes carrying an A/R PTM
- `nB`  = `n_B_nucleosome_coloc` — nucleosomes carrying a B PTM
- `nAB` = `n_triple_coloc` — nucleosomes carrying **both** (double occupancy)

giving, per nucleosome:

| class | formula |
|---|---|
| P(double)   | `nAB / N` |
| P(single A) | `(nA − nAB) / N` |
| P(single B) | `(nB − nAB) / N` |
| P(none)     | `1 − P(double) − P(single A) − P(single B) = (N − nA − nB + nAB) / N` |

Caveat: the module's coloc counts are event rows de-duplicated by nucleosome position, so a few
FOVs can drift a hair outside [0, 1]; those are flagged, clipped to 0, and renormalized so each
FOV's four probabilities sum to 1. If you set `A`/`B` to the other channel, edit `count_cols`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

OCC_CLASSES = ["p_none", "p_single_A", "p_single_B", "p_double"]
OCC_TITLES = {"p_none": "None", "p_single_A": "Single A (R)",
              "p_single_B": "Single B", "p_double": "Double"}


def occupancy_from_per_fov(per_fov, count_cols=None):
    """Per-FOV per-nucleosome occupancy probabilities (rows with N==0 dropped).

    ``count_cols`` maps the roles N/A/B/AB to column names (defaults match
    ``run_samples_per_fov``'s renamed output). Swap A/B here to flip which channel is A.
    """
    c = count_cols or dict(N="n_nucleosomes", A="n_R_nucleosome_coloc",
                           B="n_B_nucleosome_coloc", AB="n_triple_coloc")
    d = per_fov[per_fov[c["N"]] > 0].copy()
    N = d[c["N"]].astype(float)
    nAB = d[c["AB"]].astype(float)
    d["p_double"]   = nAB / N
    d["p_single_A"] = (d[c["A"]].astype(float) - nAB) / N
    d["p_single_B"] = (d[c["B"]].astype(float) - nAB) / N
    d["p_none"]     = 1.0 - d["p_double"] - d["p_single_A"] - d["p_single_B"]
    probs = d[OCC_CLASSES]
    bad = (probs < -1e-9).any(axis=1) | (probs > 1 + 1e-9).any(axis=1)
    if bad.any():
        print(f"note: {int(bad.sum())}/{len(d)} FOV(s) had out-of-range probabilities "
              f"(clipped to [0,1] and renormalized).")
    clipped = probs.clip(lower=0)
    d[OCC_CLASSES] = clipped.div(clipped.sum(axis=1), axis=0)
    d["sample_name"] = d["sample"].map(name_of)
    keep = ["sample", "sample_name", "FOV", "n_timepoints", c["N"]] + OCC_CLASSES
    return d[keep].reset_index(drop=True)


occ = occupancy_from_per_fov(df)          # df = per-FOV counts from section 2
occ.to_csv(os.path.join(OUTPUT_DIR, "per_fov_occupancy.csv"), index=False)
occ

## 5. Variability — within sample vs between samples

- **Within-sample variability**: how much each occupancy probability varies across the FOVs of
  one sample (spread of the box / the SD and CV columns).
- **Between-sample variability**: how much the sample means differ from each other.

The `variance decomposition` table puts numbers on both: `within_FOV_var` is the mean
FOV-to-FOV variance inside a sample, `between_sample_var` is the variance of the sample means,
and the one-way ANOVA tests whether the between-sample differences exceed the FOV-to-FOV noise
(small `ANOVA_p` = samples genuinely differ for that class).

In [ ]:
# Within-sample summary: mean, SD, and CV of each occupancy class across FOVs.
def _cv(x):
    m = np.mean(x)
    return np.std(x, ddof=1) / m if m else np.nan

within = occ.groupby("sample")[OCC_CLASSES].agg(["mean", "std", _cv])
within.columns = [f"{cls}_{stat.replace('_cv', 'CV')}" for cls, stat in within.columns]
within.insert(0, "n_FOVs", occ.groupby("sample")["FOV"].count())
within.insert(0, "sample_name", [name_of(s) for s in within.index])
within

In [ ]:
# Visual comparison. Each of the four panels: one box per sample (between-sample differences),
# with per-FOV points jittered on top (within-sample spread). Last panel: mean composition.
samples = sorted(occ["sample"].unique())
sample_labels = [name_of(s) for s in samples]
rng = np.random.default_rng(0)
colors = {"p_none": "#b3b3b3", "p_single_A": "#d1495b",
          "p_single_B": "#2e6f95", "p_double": "#8c4799"}

fig, axes = plt.subplots(1, len(OCC_CLASSES) + 1,
                         figsize=(3.4 * (len(OCC_CLASSES) + 1), 3.8))
for ax, cls in zip(axes, OCC_CLASSES):
    data = [occ.loc[occ["sample"] == s, cls].values for s in samples]
    ax.boxplot(data, tick_labels=sample_labels, showmeans=True,
               medianprops=dict(color="black"))
    for i, s in enumerate(samples, start=1):
        y = occ.loc[occ["sample"] == s, cls].values
        ax.scatter(i + (rng.random(len(y)) - 0.5) * 0.18, y, s=14, alpha=0.55,
                   color=colors[cls], edgecolor="none", zorder=3)
    ax.set_title(OCC_TITLES[cls]); ax.set_xlabel("sample"); ax.set_ylim(0, None)
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("probability per nucleosome")

# Stacked mean composition per sample.
axc = axes[-1]
means = occ.groupby("sample")[OCC_CLASSES].mean().loc[samples]
bottom = np.zeros(len(samples))
for cls in OCC_CLASSES:
    axc.bar(sample_labels, means[cls].values, bottom=bottom, label=OCC_TITLES[cls], color=colors[cls])
    bottom += means[cls].values
axc.set_title("Mean composition"); axc.set_xlabel("sample")
axc.set_ylabel("mean probability"); axc.set_ylim(0, 1)
axc.tick_params(axis="x", rotation=45)
axc.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "occupancy_variability.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Variance decomposition + one-way ANOVA per occupancy class.
from scipy.stats import f_oneway

rows = []
for cls in OCC_CLASSES:
    groups = [occ.loc[occ["sample"] == s, cls].values for s in samples]
    multi = [g for g in groups if len(g) > 1]
    within_var = np.mean([np.var(g, ddof=1) for g in multi]) if multi else np.nan
    between_var = np.var([g.mean() for g in groups], ddof=1) if len(groups) > 1 else np.nan
    try:
        F, p = f_oneway(*[g for g in groups if len(g) > 0])
    except Exception:
        F, p = np.nan, np.nan
    rows.append({"class": cls, "within_FOV_var": within_var,
                 "between_sample_var": between_var,
                 "between/within": (between_var / within_var) if within_var else np.nan,
                 "ANOVA_F": F, "ANOVA_p": p})

var_decomp = pd.DataFrame(rows)
var_decomp.to_csv(os.path.join(OUTPUT_DIR, "occupancy_variance_decomposition.csv"), index=False)
var_decomp

## 6. Entropy / heterogeneity measures (per FOV)

Ported from `EPINUC_MAYO12_MAYO3_Pair4_only_nobatchcorrection.ipynb`. That notebook builds a 2×2
co-occupancy table `(p00, p10, p01, p11)` from `P(A)`, `P(B)`, `P(A∩B)` — which is **exactly** the
occupancy distribution computed above:

| entropy-notebook state | here |
|---|---|
| `p00` | `p_none` |
| `p10` (A only) | `p_single_A` |
| `p01` (B only) | `p_single_B` |
| `p11` (A∩B) | `p_double` |

with `P(A) = p_single_A + p_double` and `P(B) = p_single_B + p_double`. The nine measures (same
formulas as the source `compute_features`):

- **entropy** `H` — Shannon entropy of the 4-state distribution, in bits (0 = one pure state, 2 = uniform).
- **entropy_norm** — `H / 2`, rescaled to [0, 1].
- **n_eff** — `2**H`, the effective number of occupancy states (1–4).
- **bivalent** — `p11`, the double-occupancy fraction.
- **discordance** — `p10 + p01`, the single-mark (A-xor-B) fraction.
- **polarity** — `(p10 − p01)/(p10 + p01)`, A-vs-B asymmetry in [−1, 1].
- **log2_enrichment** — `log2(P(A∩B) / (P(A)·P(B)))`, co-occurrence vs. independence.
- **mutual_info** — mutual information between A and B occupancy, in bits.
- **nmi** — `MI / min(H(A), H(B))`, normalized mutual information.

In [ ]:
# Entropy / heterogeneity feature functions (ported verbatim from the Pair-4 entropy notebook).
def safe_entropy(states, eps=1e-15):
    p = np.clip(states, eps, 1.0)
    return -np.sum(p * np.log2(p), axis=1)


def marginal_entropy(p, eps=1e-15):
    p = np.clip(p, eps, 1 - eps)
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p)


ENTROPY_FEATURES = ["entropy", "entropy_norm", "n_eff", "bivalent", "discordance",
                    "polarity", "log2_enrichment", "mutual_info", "nmi"]


def entropy_features(dist, id_cols, eps=1e-12,
                     cols=("p_none", "p_single_A", "p_single_B", "p_double")):
    """Nine heterogeneity measures on a 2x2 occupancy distribution.

    ``cols`` = (p00, p10, p01, p11) = (none, A-only, B-only, double). Works for any table with
    those four probability columns — used for both per-FOV and per-sample distributions.
    """
    p00, p10, p01, p11 = (dist[c].to_numpy(float) for c in cols)
    states = np.stack([p00, p10, p01, p11], axis=1)
    pa = p10 + p11                    # P(A) = A-only + double
    pb = p01 + p11                    # P(B) = B-only + double

    H = safe_entropy(states)
    marginals_A = np.stack([1 - pa, pa], axis=1)
    marginals_B = np.stack([1 - pb, pb], axis=1)
    joint = np.stack([p00, p01, p10, p11], axis=1).reshape(-1, 2, 2)
    outer = marginals_A[:, :, None] * marginals_B[:, None, :]
    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(outer > eps, joint / (outer + eps), 1.0)
        mi_terms = np.where(joint > eps, joint * np.log2(ratio + eps), 0.0)
    MI = mi_terms.sum(axis=(1, 2))
    HA, HB = marginal_entropy(pa), marginal_entropy(pb)
    NMI = np.where(np.minimum(HA, HB) > eps, MI / (np.minimum(HA, HB) + eps), 0.0)

    out = dist[id_cols].copy()
    out["entropy"]         = H
    out["entropy_norm"]    = H / 2.0
    out["n_eff"]           = 2.0 ** H
    out["bivalent"]        = p11
    out["discordance"]     = p10 + p01
    out["polarity"]        = (p10 - p01) / (p10 + p01 + eps)
    out["log2_enrichment"] = np.log2(np.clip(p11, eps, None) / (pa * pb + eps))
    out["mutual_info"]     = MI
    out["nmi"]             = NMI
    return out.reset_index(drop=True)


# --- Per-FOV entropy ---
FOV_ID_COLS = ["sample", "sample_name", "FOV", "n_timepoints", "n_nucleosomes"]
occ_entropy = entropy_features(occ, id_cols=FOV_ID_COLS)
occ_entropy.to_csv(os.path.join(OUTPUT_DIR, "per_fov_entropy.csv"), index=False)
occ_entropy

## 7. Entropy per sample

A sample-level occupancy distribution is built by **pooling nucleosomes across the sample's
FOVs** (count-weighted: each state's fraction = total nucleosomes in that state / total
nucleosomes), then the same nine measures are computed on it. This is the sample's overall
co-occupancy — not the average of per-FOV entropies. We also attach `mean_FOV_entropy` /
`std_FOV_entropy` (the mean and spread of the per-FOV entropies) so within-sample heterogeneity
sits next to the pooled sample value. Because entropy is concave, the pooled sample entropy is
generally ≥ the mean per-FOV entropy; the gap reflects FOV-to-FOV variability.

In [ ]:
# Sample-level occupancy = nucleosomes pooled across FOVs (count-weighted).
grp = occ.groupby("sample")
weighted = occ[OCC_CLASSES].mul(occ["n_nucleosomes"], axis=0)
weighted["sample"] = occ["sample"].values
state_tot = weighted.groupby("sample")[OCC_CLASSES].sum()
sample_dist = state_tot.div(state_tot.sum(axis=1), axis=0)   # renormalize (guards rounding)
sample_dist["n_FOVs"] = grp["FOV"].count()
sample_dist["n_nucleosomes"] = grp["n_nucleosomes"].sum()
sample_dist = sample_dist.reset_index()
sample_dist["sample_name"] = sample_dist["sample"].map(name_of)

sample_entropy = entropy_features(sample_dist, id_cols=["sample", "sample_name", "n_FOVs", "n_nucleosomes"])
# Attach within-sample per-FOV entropy mean/SD for the variability comparison.
fov_H = occ_entropy.groupby("sample")["entropy"].agg(mean_FOV_entropy="mean",
                                                     std_FOV_entropy="std").reset_index()
sample_entropy = sample_entropy.merge(fov_H, on="sample")
sample_entropy.to_csv(os.path.join(OUTPUT_DIR, "per_sample_entropy.csv"), index=False)
sample_entropy

In [ ]:
# Entropy across samples: per-FOV spread (box + points) vs the pooled sample value (red diamond).
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, feat in zip(axes, ["entropy", "n_eff"]):
    data = [occ_entropy.loc[occ_entropy["sample"] == s, feat].values for s in samples]
    ax.boxplot(data, tick_labels=sample_labels, medianprops=dict(color="black"))
    for i, s in enumerate(samples, start=1):
        y = occ_entropy.loc[occ_entropy["sample"] == s, feat].values
        ax.scatter(i + (rng.random(len(y)) - 0.5) * 0.18, y, s=14, alpha=0.55,
                   color="#2e6f95", edgecolor="none", zorder=3)
        pooled = sample_entropy.loc[sample_entropy["sample"] == s, feat].values[0]
        ax.scatter(i, pooled, marker="D", s=55, color="#d1495b", edgecolor="black",
                   zorder=5, label="pooled sample" if i == 1 else None)
    ax.set_title(feat); ax.set_xlabel("sample"); ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("bits"); axes[0].axhline(2, color="grey", ls=":", lw=0.8)  # max entropy
axes[1].set_ylabel("effective # states"); axes[1].axhline(4, color="grey", ls=":", lw=0.8)
axes[0].legend(loc="best", fontsize=8)
fig.suptitle("Occupancy entropy — per-FOV spread vs pooled sample value")
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "occupancy_entropy.png"), dpi=150, bbox_inches="tight")
plt.show()

## 8. Healthy vs sick — statistical comparison

Disease group is read from the sample name: **`H_*` = Healthy** (H_29mm, H_34mm), everything else
= **Cancer/sick** (L123, LR143, L124, LR144). So the design is **2 healthy vs 4 sick samples**,
49 FOVs each.

**Read this before trusting any p-value.** The biological replicate is the *sample*, not the FOV
— the 49 FOVs of one sample are repeated measures of the same specimen, not independent draws.
So we run two tests:

- **Per-sample (correct unit):** average each sample first, then Mann–Whitney on 2 vs 4 values.
  This is the honest test — but with n=2 vs 4 the smallest possible two-sided Mann–Whitney p is
  **2/15 ≈ 0.133**, so *nothing can reach q<0.05 no matter how large the effect*. Treat directions
  and effect sizes as suggestive, not proven.
- **Per-FOV (pseudoreplicated):** Mann–Whitney on ~98 healthy vs ~196 sick FOVs. This yields tiny
  p-values, but they overstate significance because FOVs within a sample are correlated — it tests
  "do these particular 6 specimens differ", not "do healthy and sick populations differ". Use it
  to see effect direction/magnitude, not as evidence of a population-level difference.

Bottom line up front: a properly-powered healthy-vs-sick claim needs more healthy samples.

In [ ]:
from scipy.stats import mannwhitneyu

# --- Disease group from the sample name (H_* = Healthy). Override HEALTHY_SAMPLES if needed. ---
HEALTHY_SAMPLES = {s for s in occ["sample"].unique() if name_of(s).startswith("H")}
def group_of_sample(sid):
    return "Healthy" if sid in HEALTHY_SAMPLES else "Cancer"

# Combined per-FOV feature table: occupancy classes + the non-redundant entropy measures.
ENT_TEST = ["entropy", "n_eff", "polarity", "log2_enrichment", "mutual_info", "nmi"]
FEAT_COLS = OCC_CLASSES + ENT_TEST
per_fov_feat = occ.merge(occ_entropy[["sample", "FOV"] + ENT_TEST], on=["sample", "FOV"])
per_fov_feat["group"] = per_fov_feat["sample"].map(group_of_sample)

# Per-sample means (the biological-replicate unit).
sample_feat = (per_fov_feat.groupby(["sample", "sample_name"])[FEAT_COLS].mean()
               .reset_index())
sample_feat["group"] = sample_feat["sample"].map(group_of_sample)

print("Healthy:", sorted(sample_feat.loc[sample_feat.group == "Healthy", "sample_name"]))
print("Cancer :", sorted(sample_feat.loc[sample_feat.group == "Cancer", "sample_name"]))


def bh_fdr(pvals):
    """Benjamini-Hochberg FDR q-values (no statsmodels dependency)."""
    p = np.asarray(pvals, float); n = len(p); order = np.argsort(p)
    q = (p[order] * n) / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[order] = np.clip(q, 0, 1)
    return out


def compare_groups(dfr, cols, label):
    rows = []
    for col in cols:
        h = dfr.loc[dfr.group == "Healthy", col].dropna().values
        c = dfr.loc[dfr.group == "Cancer", col].dropna().values
        U, p = mannwhitneyu(h, c, alternative="two-sided")
        rows.append({"feature": col, "n_healthy": len(h), "n_cancer": len(c),
                     "healthy_median": np.median(h), "cancer_median": np.median(c),
                     "log2_fold_median": np.log2((np.median(c) + 1e-12) / (np.median(h) + 1e-12)),
                     "mannwhitney_p": p})
    out = pd.DataFrame(rows)
    out["q_BH"] = bh_fdr(out["mannwhitney_p"].values)
    out["sig_q05"] = out["q_BH"] < 0.05
    out = out.sort_values("mannwhitney_p").reset_index(drop=True)
    print(f"\n[{label}] significant at BH q<0.05: {int(out['sig_q05'].sum())}/{len(out)}")
    return out

In [ ]:
# (A) PER-SAMPLE test — the statistically correct unit (n=2 healthy vs 4 sick).
# NOTE: min possible two-sided Mann-Whitney p at 2-vs-4 is 2/15 = 0.133, so q<0.05 is unreachable.
sample_stats = compare_groups(sample_feat, FEAT_COLS, "per-sample (n=2 vs 4)")
sample_stats.to_csv(os.path.join(OUTPUT_DIR, "healthy_vs_cancer_per_sample.csv"), index=False)
sample_stats

In [ ]:
# (B) PER-FOV test — pseudoreplicated (~98 vs ~196 FOVs). Small p-values here are INFLATED:
# they show effect direction/size, not a population-level healthy-vs-sick difference.
fov_stats = compare_groups(per_fov_feat, FEAT_COLS, "per-FOV (pseudoreplicated)")
fov_stats.to_csv(os.path.join(OUTPUT_DIR, "healthy_vs_cancer_per_fov.csv"), index=False)
fov_stats

In [ ]:
# Visualize: per-FOV distributions by group, with each sample's mean overlaid so the
# pseudoreplication (points clustering by specimen) is visible. Big markers = sample means.
GROUP_COLOR = {"Healthy": "#4dac26", "Cancer": "#d01c8b"}
show = FEAT_COLS
ncol = 5
nrow = int(np.ceil(len(show) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.2 * nrow))
axes = np.atleast_1d(axes).flatten()
groups = ["Healthy", "Cancer"]
for ax, col in zip(axes, show):
    for gi, g in enumerate(groups):
        fv = per_fov_feat.loc[per_fov_feat.group == g, col].values
        ax.scatter(np.full(len(fv), gi) + (rng.random(len(fv)) - 0.5) * 0.22, fv,
                   s=8, alpha=0.25, color=GROUP_COLOR[g], edgecolor="none", zorder=1)
        # each sample's mean as a large diamond
        for s in sorted(sample_feat.loc[sample_feat.group == g, "sample"]):
            m = sample_feat.loc[sample_feat["sample"] == s, col].values[0]
            ax.scatter(gi, m, marker="D", s=70, color=GROUP_COLOR[g], edgecolor="black",
                       linewidth=0.8, zorder=3)
    pj = sample_stats.loc[sample_stats.feature == col, "mannwhitney_p"].values[0]
    pf = fov_stats.loc[fov_stats.feature == col, "mannwhitney_p"].values[0]
    ax.set_title(f"{col}\nsample p={pj:.3f} | FOV p={pf:.1e}", fontsize=8)
    ax.set_xticks([0, 1]); ax.set_xticklabels(groups)
for ax in axes[len(show):]:
    ax.set_visible(False)
fig.suptitle("Healthy vs sick — per-FOV points (faint) and per-sample means (diamonds)", y=1.005)
fig.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "healthy_vs_cancer.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# At n=2 vs 4 a p-value is uninformative (floored at 0.133). A better readout: does EVERY sick
# sample fall on the same side of EVERY healthy sample (complete group separation)? That is the
# strongest signal achievable here and flags the most promising markers for a larger study.
rows = []
for col in FEAT_COLS:
    h = sample_feat.loc[sample_feat.group == "Healthy", col].values
    c = sample_feat.loc[sample_feat.group == "Cancer", col].values
    if c.min() > h.max():
        sep, gap = "Cancer > Healthy (all)", c.min() - h.max()
    elif c.max() < h.min():
        sep, gap = "Cancer < Healthy (all)", h.min() - c.max()
    else:
        sep, gap = "overlap", 0.0
    rows.append({"feature": col,
                 "healthy_range": f"{h.min():.4g} – {h.max():.4g}",
                 "cancer_range": f"{c.min():.4g} – {c.max():.4g}",
                 "separation": sep, "gap": gap})
separation = pd.DataFrame(rows).sort_values(["separation", "gap"], ascending=[True, False])
separation.to_csv(os.path.join(OUTPUT_DIR, "healthy_vs_cancer_separation.csv"), index=False)
separation

### Result (this dataset)

**No difference is statistically significant at the proper unit of analysis.** Comparing the 2
healthy vs 4 sick *samples*, every feature gives Mann–Whitney p ≥ 0.133 (BH q ≥ 0.38) — and that
0.133 is the mathematical floor for a 2-vs-4 test, so significance was unreachable regardless of
effect size.

**But there is a consistent, promising signal.** For three coupling/co-occupancy measures —
**`p_double` (double occupancy), `mutual_info`, and `nmi`** — *every sick sample exceeds both
healthy samples* (complete separation; the best a 2-vs-4 design can show). Sick nucleosomes carry
both marks together more often, and the two marks are more statistically coupled, than in healthy.
`entropy`, `n_eff`, and `p_single_B` trend the same way (cancer higher) but with one overlap each.

**The per-FOV test is misleading here.** It flags 9/10 features at q < 1e-20, but that significance
is pseudoreplication — it reflects these 6 specimens, not the healthy/sick populations. Note one
healthy sample (H_29mm) overlaps the lower cancer samples on entropy, which is exactly why the
honest per-sample test can't separate the groups.

**Conclusion:** suggestive that cancer shows higher A/B co-occupancy and mark coupling, but **not
proven** — 2 healthy samples cannot support a significance claim. Prioritize `p_double` /
`mutual_info` / `nmi` and repeat with more healthy specimens (≈5+/group for a properly powered test).